# Session 15 — Retrieval-Augmented Generation, Properly

**The concept:** LLMs don't know your data, and fine-tuning is the wrong
tool for fresh, regulated, or private facts. So retrieve at inference time.

**The pipeline:** ingest → chunk → embed → index → retrieve → rerank →
assemble → generate → cite.

**The lesson:** *retrieval quality dominates generation quality.*

**No API key and no network required.** The corpus ships with the repo.


In [ ]:
import json, os, sys, time, pathlib
sys.path.insert(0, os.path.join('..', 'src'))
print('python', sys.version.split()[0])


## 1. Load the corpus


In [ ]:
from vectorstore import load_corpus
from embeddings_local import EmbeddingBackend

backend = EmbeddingBackend(mode='auto', model='bge-m3', url='http://127.0.0.1:11434')
try:
    backend.embed('probe')
    print('backend resolved:', backend.resolved)
except Exception as exc:
    print('no local Ollama -- falling back to hash embeddings:', exc)

store = load_corpus('../data/corpus', embed_fn=backend.embed)
print('documents loaded:', len(store))


## 2. Three ways to search


In [ ]:
q = 'How do I stop an agent from looping forever?'

for mode in ('dense', 'lexical', 'hybrid'):
    print(f'--- {mode} ---')
    for r in store.search(q, k=3, mode=mode):
        print(f'  {r.score:8.4f}  {r.chunk.id:22s} {r.chunk.title}')


## 3. Measure it — Recall@k and MRR

If retrieval quality dominates, then retrieval needs a number. The corpus
ships with a hand-labeled question set.


In [ ]:
with open('../data/questions.json') as fh:
    questions = json.load(fh)

def measure(mode, rerank=False, k=10):
    hits1 = 0; mrr = 0.0
    for qq in questions:
        results = store.search(qq['question'], k=k, mode=mode, rerank=rerank)
        ids = [r.chunk.id for r in results]
        if qq['expected_doc_id'] in ids:
            mrr += 1.0 / (ids.index(qq['expected_doc_id']) + 1)
            if ids[0] == qq['expected_doc_id']:
                hits1 += 1
    n = len(questions)
    return hits1 / n, mrr / n

for mode in ('dense', 'lexical', 'hybrid'):
    r1, mr = measure(mode)
    print(f'  {mode:8s} Recall@1={r1:.3f}  MRR={mr:.3f}')


## 4. Reranking — narrow the candidate pool, then score it carefully

A first-stage retriever casts a wide net. A reranker scores those
candidates more carefully. Narrowing fifty down to five often changes the
final answer more than switching to a ten-times-larger generator.


In [ ]:
for mode in ('dense', 'lexical', 'hybrid'):
    r0, m0 = measure(mode, rerank=False)
    r1, m1 = measure(mode, rerank=True)
    print(f'  {mode:8s} plain R@1={r0:.3f} MRR={m0:.3f}  ->  rerank R@1={r1:.3f} MRR={m1:.3f}  (dMRR={m1-m0:+.3f})')


## 5. The headline: embedding quality is retrieval quality

Same corpus. Same queries. Same code. **Only the embedding model changed.**

The hash fallback understands character overlap. `bge-m3` understands
meaning. That gap is the entire session.


In [ ]:
from embeddings_local import hash_embedding

hash_store = load_corpus('../data/corpus', embed_fn=hash_embedding)

def top1(store_):
    hits = 0
    for qq in questions:
        rs = store_.search(qq['question'], k=1, mode='dense')
        if rs and rs[0].chunk.id == qq['expected_doc_id']:
            hits += 1
    return hits / len(questions)

h, r = top1(hash_store), top1(store)
print(f'  hash embeddings : top-1 accuracy {h:.1%}')
print(f'  real embeddings : top-1 accuracy {r:.1%}')
print()
print('A better embedder beat a bigger model. Retrieval quality dominates.')


## Takeaway

- Chunk, embed, retrieve, **rerank**, assemble, cite.
- **Measure retrieval.** Recall@k and MRR, not vibes.
- Hybrid search beats either dense or lexical alone.
- Reranking is often the cheapest quality win available.
- **Retrieval quality dominates generation quality.**
